<a href="https://colab.research.google.com/github/alsammachm/adaptive-test-time-reasoning/blob/main/notebooks/adaptive_test_time_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Jul 30 12:36:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import datasets
    print("Datasets:", datasets.__version__)
except Exception as error:
    print("Datasets error:", type(error).__name__, error)

try:
    import bitsandbytes
    print("BitsAndBytes:", bitsandbytes.__version__)
except Exception as error:
    print("BitsAndBytes error:", type(error).__name__, error)

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 5.13.1
CUDA available: True
GPU: Tesla T4
Datasets: 4.0.0
BitsAndBytes error: ModuleNotFoundError No module named 'bitsandbytes'


In [ ]:
%pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.8 MB/s eta 0:00:00


In [ ]:
import bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit

print("BitsAndBytes:", bnb.__version__)
print("4-bit quantization available:", Linear4bit is not None)

BitsAndBytes: 0.50.0
4-bit quantization available: True


In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN was not found or notebook access is disabled.")

profile = whoami(token=hf_token)

print("Authenticated as:", profile["name"])
print("Token verification: successful")

Authenticated as: alsammachm
Token verification: successful


In [ ]:
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3.5-4B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

print("Model repository:", MODEL_ID)
print("Model type:", config.model_type)
print("Architecture:", config.architectures)
print("Model access: successful")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

Model repository: Qwen/Qwen3.5-4B
Model type: qwen3_5
Architecture: ['Qwen3_5ForConditionalGeneration']
Model access: successful


In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

print("Processor class:", type(processor).__name__)
print("Tokenizer class:", type(processor.tokenizer).__name__)
print("Vocabulary size:", len(processor.tokenizer))
print("Processor loaded successfully")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Processor class: Qwen3VLProcessor
Tokenizer class: Qwen2Tokenizer
Vocabulary size: 248077
Processor loaded successfully


In [ ]:
import gc
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="auto",
    quantization_config=quantization_config,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

print("Model class:", type(model).__name__)
print("Primary device:", model.device)
print(
    "Model memory footprint:",
    f"{model.get_memory_footprint() / (1024 ** 3):.2f} GB",
)
print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB",
)
print(
    "GPU memory reserved:",
    f"{torch.cuda.memory_reserved() / (1024 ** 3):.2f} GB",
)
print("Model loaded successfully")

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Model class: Qwen3_5ForConditionalGeneration
Primary device: cuda:0
Model memory footprint: 3.01 GB
GPU memory allocated: 3.08 GB
GPU memory reserved: 3.12 GB
Model loaded successfully


In [ ]:
import time
import torch

test_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "A book costs $24 after a 20% discount. "
                    "What was its original price? "
                    "Please reason step by step, and put your final answer within \\boxed{}."
                ),
            }
        ],
    }
]

test_inputs = processor.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_length = test_inputs["input_ids"].shape[-1]

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    test_outputs = model.generate(
        **test_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

generated_tokens = test_outputs[0][input_length:]
response = processor.decode(
    generated_tokens,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

print("Generated tokens:", len(generated_tokens))
print("Generation time:", f"{elapsed_seconds:.2f} seconds")
print("\nMODEL OUTPUT\n")
print(response)

Generated tokens: 256
Generation time: 25.14 seconds

MODEL OUTPUT

Here's my thought process for solving this problem:

1.  **Analyze the Problem:**
    *   Given: The price of a book *after* a discount is $24.
    *   Given: The discount rate is 20%.
    *   Goal: Find the original price of the book before the discount.

2.  **Define Variables:**
    *   Let $P$ be the original price.
    *   Let $D$ be the discount amount.
    *   Let $S$ be the sale price (price after discount).
    *   We know $S = 24$.
    *   We know the discount rate is $20\%$.

3.  **Formulate Equations:**
    *   The relationship between original price, discount, and sale price is: $P - D = S$.
    *   The discount amount $D$ is calculated as a percentage of the original price $P$. So, $D = r \times P$, where $r$ is the rate as a decimal.
    *   Here, $r = 20\% = 0.20$.
    *   So,


In [ ]:
import time
import torch

# Fix the random seed so this run can be reproduced.
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    test_outputs_512 = model.generate(
        **test_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
elapsed_seconds_512 = time.perf_counter() - start_time

generated_tokens_512 = test_outputs_512[0][input_length:]

response_512 = processor.decode(
    generated_tokens_512,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

eos_ids = model.generation_config.eos_token_id

if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
elif eos_ids is None:
    eos_ids = []

last_token_id = int(generated_tokens_512[-1])
finished_naturally = last_token_id in eos_ids

print("Token budget:", 512)
print("Generated tokens:", len(generated_tokens_512))
print("Generation time:", f"{elapsed_seconds_512:.2f} seconds")
print("Finished naturally:", finished_naturally)
print("\nMODEL OUTPUT\n")
print(response_512)

Token budget: 512
Generated tokens: 512
Generation time: 47.60 seconds
Finished naturally: False

MODEL OUTPUT

Here's my thought process for solving this problem:

1.  **Analyze the Request:**
    *   **Given:** The final price of a book is $\$24$. The discount applied is $20\%$.
    *   **Goal:** Find the original price (before the discount).
    *   **Constraint:** Reason step by step and put the final answer in `\boxed{}`.

2.  **Understand the Mathematical Relationship:**
    *   Let $P$ be the original price.
    *   Let $d$ be the discount rate ($20\%$ or $0.20$).
    *   Let $C$ be the final price after discount ($\$24$).
    *   The formula connecting these variables is: $C = P - (\text{discount rate} \times P)$ or $C = P(1 - \text{discount rate})$.

3.  **Perform the Calculation:**
    *   Method 1: Using the percentage relationship.
        *   The final price is $20\%$ less than the original price.
        *   This means the final price represents $100\% - 20\% = 80\%$ of t

In [ ]:
import re
import time
import torch

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    test_outputs_1024 = model.generate(
        **test_inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
elapsed_seconds_1024 = time.perf_counter() - start_time

generated_tokens_1024 = test_outputs_1024[0][input_length:]

response_1024 = processor.decode(
    generated_tokens_1024,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

eos_ids = model.generation_config.eos_token_id

if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
elif eos_ids is None:
    eos_ids = []

finished_naturally_1024 = int(generated_tokens_1024[-1]) in eos_ids

boxed_answers = re.findall(
    r"\\boxed\{([^{}]+)\}",
    response_1024,
)

print("Token budget:", 1024)
print("Generated tokens:", len(generated_tokens_1024))
print("Generation time:", f"{elapsed_seconds_1024:.2f} seconds")
print("Finished naturally:", finished_naturally_1024)
print("Boxed answers found:", boxed_answers)
print("\nFINAL 2,000 CHARACTERS OF OUTPUT\n")
print(response_1024[-2000:])

Token budget: 1024
Generated tokens: 1024
Generation time: 87.80 seconds
Finished naturally: False
Boxed answers found: []

FINAL 2,000 CHARACTERS OF OUTPUT

 - 20\% = 80\%$ of the original price.
        *   So, $0.80 \times P = 24$.
        *   To find $P$, I need to divide $24$ by $0.80$.
        *   $P = 24 / 0.80$.
        *   $P = 24 / (8/10) = 24 \times (10/8) = 24 \times 1.25$.
        *   Calculation:
            *   $24 / 8 = 3$.
            *   $3 \times 10 = 30$.
            *   $30 \times 1.25$... wait, that's confusing.
            *   Let's stick to the fraction multiplication: $24 \times 1.25 = 24 + (0.25 \times 24)$.
            *   $0.25$ of $24$ is $6$.
            *   $24 + 6 = 30$.
    *   Method 2: Using the discount amount.
        *   Final Price = $24$.
        *   Discount rate = $0.20$ (20%).
        *   Final Price represents $80\%$ of the original.
        *   Original Price $\times 0.8 = 24$.
        *   Original Price $= 24 / 0.8$.
        *   $24 / 0.8 =

In [ ]:
import re
import time
import torch

baseline_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "A book costs $24 after a 20% discount. "
                    "What was its original price? "
                    "Return only the final numeric answer inside \\boxed{}."
                ),
            }
        ],
    }
]

baseline_inputs = processor.apply_chat_template(
    baseline_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

baseline_input_length = baseline_inputs["input_ids"].shape[-1]

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    baseline_outputs = model.generate(
        **baseline_inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
baseline_elapsed = time.perf_counter() - start_time

baseline_tokens = baseline_outputs[0][baseline_input_length:]

baseline_response = processor.decode(
    baseline_tokens,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

boxed_answers = re.findall(
    r"\\boxed\{([^{}]+)\}",
    baseline_response,
)

print("Mode: non-thinking")
print("Token budget: 64")
print("Generated tokens:", len(baseline_tokens))
print("Generation time:", f"{baseline_elapsed:.2f} seconds")
print("Boxed answers found:", boxed_answers)
print("\nMODEL OUTPUT\n")
print(baseline_response)

Mode: non-thinking
Token budget: 64
Generated tokens: 64
Generation time: 8.06 seconds
Boxed answers found: []

MODEL OUTPUT

Let $P$ be the original price of the book.
A 20% discount means the customer pays 80% of the original price.
So, the discounted price is $0.80 \times P$.

We are given that the discounted price is $24.
$$0


In [ ]:
import re
import time
import torch

strict_baseline_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": (
                    "You are an answer-only mathematics solver. "
                    "Return exactly one line in the format \\boxed{number}. "
                    "Do not provide reasoning, explanation, equations, labels, "
                    "or any additional text."
                ),
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "A book costs $24 after a 20% discount. "
                    "What was its original price?"
                ),
            }
        ],
    },
]

strict_inputs = processor.apply_chat_template(
    strict_baseline_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

strict_input_length = strict_inputs["input_ids"].shape[-1]

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    strict_outputs = model.generate(
        **strict_inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
strict_elapsed = time.perf_counter() - start_time

strict_tokens = strict_outputs[0][strict_input_length:]

strict_response = processor.decode(
    strict_tokens,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

boxed_answers = re.findall(
    r"\\boxed\{([^{}]+)\}",
    strict_response,
)

eos_ids = model.generation_config.eos_token_id

if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
elif eos_ids is None:
    eos_ids = []

finished_naturally = (
    len(strict_tokens) > 0
    and int(strict_tokens[-1]) in eos_ids
)

print("Mode: strict non-thinking")
print("Token budget: 128")
print("Generated tokens:", len(strict_tokens))
print("Generation time:", f"{strict_elapsed:.2f} seconds")
print("Finished naturally:", finished_naturally)
print("Boxed answers found:", boxed_answers)
print("\nMODEL OUTPUT\n")
print(strict_response)

Mode: strict non-thinking
Token budget: 128
Generated tokens: 9
Generation time: 2.55 seconds
Finished naturally: True
Boxed answers found: ['30']

MODEL OUTPUT

\boxed{30}<|im_end|>
<|endoftext|>


In [ ]:
from datasets import load_dataset

gsm8k_test = load_dataset(
    "openai/gsm8k",
    "main",
    split="test",
)

print("Dataset loaded successfully")
print("Number of test questions:", len(gsm8k_test))
print("Available fields:", gsm8k_test.column_names)

print("\nFIRST QUESTION\n")
print(gsm8k_test[0]["question"])

print("\nREFERENCE ANSWER\n")
print(gsm8k_test[0]["answer"])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully
Number of test questions: 1319
Available fields: ['question', 'answer']

FIRST QUESTION

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

REFERENCE ANSWER

Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [ ]:
import random
import re
import pandas as pd

PILOT_SIZE = 20
RANDOM_SEED = 2026

random_generator = random.Random(RANDOM_SEED)

pilot_indices = sorted(
    random_generator.sample(
        range(len(gsm8k_test)),
        PILOT_SIZE,
    )
)

pilot_dataset = gsm8k_test.select(pilot_indices)


def extract_reference_answer(answer_text: str) -> str:
    """
    Extract the final GSM8K answer appearing after ####.
    """
    match = re.search(
        r"####\s*(-?[\d,]+(?:\.\d+)?)",
        answer_text,
    )

    if match is None:
        raise ValueError(
            f"Could not extract reference answer from: {answer_text}"
        )

    return match.group(1).replace(",", "")


pilot_records = []

for pilot_position, example in enumerate(pilot_dataset):
    pilot_records.append(
        {
            "pilot_position": pilot_position,
            "gsm8k_index": pilot_indices[pilot_position],
            "question": example["question"],
            "reference_answer": extract_reference_answer(
                example["answer"]
            ),
        }
    )

pilot_df = pd.DataFrame(pilot_records)

print("Pilot sample created successfully")
print("Random seed:", RANDOM_SEED)
print("Number of questions:", len(pilot_df))
print("Selected GSM8K indices:", pilot_indices)

display(
    pilot_df[
        [
            "pilot_position",
            "gsm8k_index",
            "question",
            "reference_answer",
        ]
    ].head(5)
)

Pilot sample created successfully
Random seed: 2026
Number of questions: 20
Selected GSM8K indices: [5, 165, 210, 226, 243, 457, 491, 654, 861, 903, 1005, 1029, 1048, 1121, 1139, 1172, 1201, 1230, 1257, 1272]


,pilot_position,gsm8k_index,question,reference_answer
0,0,5,Kylar went to the store to buy glasses for his...,64
1,1,165,"For his 30th birthday, Elvira chose a new comp...",77
2,2,210,Sara wants to buy herself a new jacket and 2 p...,10
3,3,226,James is counting his Pokemon cards. He has 30...,33
4,4,243,Jason has a phone plan of 1000 minutes per mon...,250


In [ ]:
import re
import time
import torch
from decimal import Decimal, InvalidOperation
from fractions import Fraction


STRICT_SYSTEM_PROMPT = (
    "You are an answer-only mathematics solver. "
    "Return exactly one line in the format \\boxed{number}. "
    "Do not provide reasoning, explanation, equations, labels, "
    "units, or any additional text."
)


def normalize_numeric_answer(value):
    """
    Convert common integer, decimal, and fraction formats
    into Decimal values for reliable comparison.
    """
    if value is None:
        return None

    cleaned = (
        str(value)
        .strip()
        .replace(",", "")
        .replace("$", "")
        .replace("\\,", "")
        .replace(" ", "")
    )

    cleaned = cleaned.rstrip(".")

    fraction_match = re.fullmatch(
        r"(-?\d+)\s*/\s*(-?\d+)",
        cleaned,
    )

    if fraction_match:
        numerator = int(fraction_match.group(1))
        denominator = int(fraction_match.group(2))

        if denominator == 0:
            return None

        fraction = Fraction(numerator, denominator)

        return (
            Decimal(fraction.numerator)
            / Decimal(fraction.denominator)
        )

    try:
        return Decimal(cleaned)
    except InvalidOperation:
        return None


def answers_match(predicted, reference):
    predicted_value = normalize_numeric_answer(predicted)
    reference_value = normalize_numeric_answer(reference)

    if predicted_value is None or reference_value is None:
        return False

    tolerance = Decimal("0.000000001")

    return abs(predicted_value - reference_value) <= tolerance


def solve_strict_non_thinking(
    question,
    max_new_tokens=128,
):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": STRICT_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question,
                }
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_length:]

    response = processor.decode(
        generated_tokens,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    boxed_answers = re.findall(
        r"\\boxed\{([^{}]+)\}",
        response,
    )

    predicted_answer = (
        boxed_answers[-1]
        if boxed_answers
        else None
    )

    eos_ids = model.generation_config.eos_token_id

    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    elif eos_ids is None:
        eos_ids = []

    finished_naturally = (
        len(generated_tokens) > 0
        and int(generated_tokens[-1]) in eos_ids
    )

    return {
        "predicted_answer": predicted_answer,
        "raw_response": response,
        "generated_tokens": len(generated_tokens),
        "elapsed_seconds": elapsed_seconds,
        "finished_naturally": finished_naturally,
    }


sample = pilot_df.iloc[0]

sample_result = solve_strict_non_thinking(
    question=sample["question"],
)

sample_correct = answers_match(
    sample_result["predicted_answer"],
    sample["reference_answer"],
)

print("GSM8K index:", sample["gsm8k_index"])
print("\nQUESTION\n")
print(sample["question"])
print("\nReference answer:", sample["reference_answer"])
print("Predicted answer:", sample_result["predicted_answer"])
print("Correct:", sample_correct)
print("Generated tokens:", sample_result["generated_tokens"])
print(
    "Generation time:",
    f"{sample_result['elapsed_seconds']:.2f} seconds",
)
print(
    "Finished naturally:",
    sample_result["finished_naturally"],
)
print("\nRAW MODEL OUTPUT\n")
print(sample_result["raw_response"])

GSM8K index: 5

QUESTION

Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?

Reference answer: 64
Predicted answer: 68
Correct: False
Generated tokens: 9
Generation time: 4.03 seconds
Finished naturally: True

RAW MODEL OUTPUT

\boxed{68}<|im_end|>
<|endoftext|>


In [ ]:
import re
import time
import torch

THINKING_SYSTEM_PROMPT = (
    "Solve the mathematics problem carefully. "
    "Use concise reasoning and verify the calculation. "
    "End with the final numeric answer in the format \\boxed{number}."
)

thinking_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": THINKING_SYSTEM_PROMPT,
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": sample["question"],
            }
        ],
    },
]

thinking_inputs = processor.apply_chat_template(
    thinking_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)

thinking_input_length = thinking_inputs["input_ids"].shape[-1]

torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    thinking_outputs = model.generate(
        **thinking_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
thinking_elapsed = time.perf_counter() - start_time

thinking_tokens = thinking_outputs[0][thinking_input_length:]

thinking_response = processor.decode(
    thinking_tokens,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

boxed_answers = re.findall(
    r"\\boxed\{([^{}]+)\}",
    thinking_response,
)

predicted_answer = (
    boxed_answers[-1]
    if boxed_answers
    else None
)

eos_ids = model.generation_config.eos_token_id

if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
elif eos_ids is None:
    eos_ids = []

finished_naturally = (
    len(thinking_tokens) > 0
    and int(thinking_tokens[-1]) in eos_ids
)

correct = answers_match(
    predicted_answer,
    sample["reference_answer"],
)

print("Mode: thinking")
print("Token budget: 512")
print("Reference answer:", sample["reference_answer"])
print("Predicted answer:", predicted_answer)
print("Correct:", correct)
print("Generated tokens:", len(thinking_tokens))
print("Generation time:", f"{thinking_elapsed:.2f} seconds")
print("Finished naturally:", finished_naturally)
print("\nFINAL 1,500 CHARACTERS OF OUTPUT\n")
print(thinking_response[-1500:])

Mode: thinking
Token budget: 512
Reference answer: 64
Predicted answer: None
Correct: False
Generated tokens: 512
Generation time: 67.01 seconds
Finished naturally: False

FINAL 1,500 CHARACTERS OF OUTPUT

The user wants me to solve a math problem about buying glasses.

**1. Understand the pricing structure:**
*   Price of the 1st glass: $5
*   Price of every second glass: 60% of the price of a regular glass.
*   "Every second glass" implies a pattern: 1st (regular), 2nd (discounted), 3rd (regular), 4th (discounted), etc.
*   Regular price = $5
*   Discounted price = 60% of $5

**2. Calculate the discounted price:**
*   $5 \times 0.60 = $3.00

**3. Analyze the quantity:**
*   Kylar wants to buy 16 glasses.

**4. Determine the cost for each glass in the set of 16:**
*   Glasses are bought in pairs (1st, 2nd, 3rd, 4th...).
*   The 1st glass costs $5.
*   The 2nd glass costs $3.
*   The 3rd glass costs $5.
*   The 4th glass costs $3.
*   ...
*   The 15th glass costs $5.
*   The 16th glass

In [ ]:
import re
import time
import torch

FINALIZER_SYSTEM_PROMPT = (
    "You are a strict mathematical answer extractor. "
    "Use the supplied reasoning to determine the final numeric answer. "
    "Return exactly one line in the format \\boxed{number}. "
    "Do not include any explanation or additional text."
)

finalizer_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": FINALIZER_SYSTEM_PROMPT,
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    f"Problem:\n{sample['question']}\n\n"
                    f"Reasoning trace:\n{thinking_response}\n\n"
                    "Extract the final numeric answer."
                ),
            }
        ],
    },
]

finalizer_inputs = processor.apply_chat_template(
    finalizer_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

finalizer_input_length = finalizer_inputs["input_ids"].shape[-1]

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    finalizer_outputs = model.generate(
        **finalizer_inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
finalizer_elapsed = time.perf_counter() - start_time

finalizer_tokens = finalizer_outputs[0][finalizer_input_length:]

finalizer_response = processor.decode(
    finalizer_tokens,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

boxed_answers = re.findall(
    r"\\boxed\{([^{}]+)\}",
    finalizer_response,
)

final_answer = (
    boxed_answers[-1]
    if boxed_answers
    else None
)

final_correct = answers_match(
    final_answer,
    sample["reference_answer"],
)

print("Reference answer:", sample["reference_answer"])
print("Final extracted answer:", final_answer)
print("Correct:", final_correct)
print("Reasoning tokens:", len(thinking_tokens))
print("Finalizer tokens:", len(finalizer_tokens))
print(
    "Combined generated tokens:",
    len(thinking_tokens) + len(finalizer_tokens),
)
print(
    "Combined generation time:",
    f"{thinking_elapsed + finalizer_elapsed:.2f} seconds",
)
print("\nFINALIZER OUTPUT\n")
print(finalizer_response)

Reference answer: 64
Final extracted answer: 64
Correct: True
Reasoning tokens: 512
Finalizer tokens: 9
Combined generated tokens: 521
Combined generation time: 70.51 seconds

FINALIZER OUTPUT

\boxed{64}<|im_end|>
<|endoftext|>


In [1]:
import time
import pandas as pd
from tqdm.auto import tqdm

baseline_results = []

experiment_start = time.perf_counter()

for _, row in tqdm(
    pilot_df.iterrows(),
    total=len(pilot_df),
    desc="Running strict baseline",
):
    result = solve_strict_non_thinking(
        question=row["question"],
        max_new_tokens=128,
    )

    correct = answers_match(
        result["predicted_answer"],
        row["reference_answer"],
    )

    baseline_results.append(
        {
            "pilot_position": int(row["pilot_position"]),
            "gsm8k_index": int(row["gsm8k_index"]),
            "question": row["question"],
            "reference_answer": row["reference_answer"],
            "predicted_answer": result["predicted_answer"],
            "correct": correct,
            "generated_tokens": result["generated_tokens"],
            "elapsed_seconds": result["elapsed_seconds"],
            "finished_naturally": result["finished_naturally"],
            "raw_response": result["raw_response"],
        }
    )

    # Save after every question so progress survives an interruption.
    pd.DataFrame(baseline_results).to_csv(
        "/content/strict_baseline_results.csv",
        index=False,
    )

baseline_results_df = pd.DataFrame(baseline_results)

total_runtime = time.perf_counter() - experiment_start
accuracy = baseline_results_df["correct"].mean()
average_tokens = baseline_results_df["generated_tokens"].mean()
average_time = baseline_results_df["elapsed_seconds"].mean()

print("Strict non-thinking baseline completed")
print("Questions evaluated:", len(baseline_results_df))
print("Correct answers:", int(baseline_results_df["correct"].sum()))
print("Accuracy:", f"{accuracy:.1%}")
print("Average generated tokens:", f"{average_tokens:.1f}")
print("Average time per question:", f"{average_time:.2f} seconds")
print("Total experiment time:", f"{total_runtime / 60:.2f} minutes")
print(
    "Naturally completed:",
    f"{baseline_results_df['finished_naturally'].mean():.1%}",
)

display(
    baseline_results_df[
        [
            "pilot_position",
            "gsm8k_index",
            "reference_answer",
            "predicted_answer",
            "correct",
            "generated_tokens",
            "elapsed_seconds",
        ]
    ]
)

NameError: name 'pilot_df' is not defined

In [2]:
# Fresh-runtime bootstrap for the adaptive reasoning experiment.
# This cell recreates everything required by the next experiment cells.

%pip install -q bitsandbytes==0.50.0

import gc
import random
import re
import time
import torch
import pandas as pd

from decimal import Decimal, InvalidOperation
from fractions import Fraction
from google.colab import userdata
from datasets import load_dataset
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

# -------------------------------------------------------------------
# 1. Verify GPU and Hugging Face authentication
# -------------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not active. Select Runtime > Change runtime type > T4 GPU."
    )

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN was not found. Confirm that the Colab secret exists "
        "and Notebook access is enabled."
    )

print("GPU:", torch.cuda.get_device_name(0))

# -------------------------------------------------------------------
# 2. Load the processor and quantized reasoning model
# -------------------------------------------------------------------

MODEL_ID = "Qwen/Qwen3.5-4B"

gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="auto",
    quantization_config=quantization_config,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

# -------------------------------------------------------------------
# 3. Load the benchmark and recreate the fixed pilot sample
# -------------------------------------------------------------------

gsm8k_test = load_dataset(
    "openai/gsm8k",
    "main",
    split="test",
)

pilot_indices = [
    5, 165, 210, 226, 243,
    457, 491, 654, 861, 903,
    1005, 1029, 1048, 1121, 1139,
    1172, 1201, 1230, 1257, 1272,
]


def extract_reference_answer(answer_text: str) -> str:
    match = re.search(
        r"####\s*(-?[\d,]+(?:\.\d+)?)",
        answer_text,
    )

    if match is None:
        raise ValueError(
            f"Could not extract reference answer from: {answer_text}"
        )

    return match.group(1).replace(",", "")


pilot_dataset = gsm8k_test.select(pilot_indices)

pilot_records = []

for pilot_position, example in enumerate(pilot_dataset):
    pilot_records.append(
        {
            "pilot_position": pilot_position,
            "gsm8k_index": pilot_indices[pilot_position],
            "question": example["question"],
            "reference_answer": extract_reference_answer(
                example["answer"]
            ),
        }
    )

pilot_df = pd.DataFrame(pilot_records)

# -------------------------------------------------------------------
# 4. Recreate answer comparison utilities
# -------------------------------------------------------------------

def normalize_numeric_answer(value):
    if value is None:
        return None

    cleaned = (
        str(value)
        .strip()
        .replace(",", "")
        .replace("$", "")
        .replace("\\,", "")
        .replace(" ", "")
        .rstrip(".")
    )

    fraction_match = re.fullmatch(
        r"(-?\d+)\s*/\s*(-?\d+)",
        cleaned,
    )

    if fraction_match:
        numerator = int(fraction_match.group(1))
        denominator = int(fraction_match.group(2))

        if denominator == 0:
            return None

        fraction = Fraction(numerator, denominator)

        return (
            Decimal(fraction.numerator)
            / Decimal(fraction.denominator)
        )

    try:
        return Decimal(cleaned)
    except InvalidOperation:
        return None


def answers_match(predicted, reference):
    predicted_value = normalize_numeric_answer(predicted)
    reference_value = normalize_numeric_answer(reference)

    if predicted_value is None or reference_value is None:
        return False

    tolerance = Decimal("0.000000001")

    return abs(predicted_value - reference_value) <= tolerance


# -------------------------------------------------------------------
# 5. Recreate the strict non-thinking baseline solver
# -------------------------------------------------------------------

STRICT_SYSTEM_PROMPT = (
    "You are an answer-only mathematics solver. "
    "Return exactly one line in the format \\boxed{number}. "
    "Do not provide reasoning, explanation, equations, labels, "
    "units, or any additional text."
)


def solve_strict_non_thinking(
    question,
    max_new_tokens=128,
):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": STRICT_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question,
                }
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_length:]

    response = processor.decode(
        generated_tokens,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    boxed_answers = re.findall(
        r"\\boxed\{([^{}]+)\}",
        response,
    )

    predicted_answer = (
        boxed_answers[-1]
        if boxed_answers
        else None
    )

    eos_ids = model.generation_config.eos_token_id

    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    elif eos_ids is None:
        eos_ids = []

    finished_naturally = (
        len(generated_tokens) > 0
        and int(generated_tokens[-1]) in eos_ids
    )

    return {
        "predicted_answer": predicted_answer,
        "raw_response": response,
        "generated_tokens": len(generated_tokens),
        "elapsed_seconds": elapsed_seconds,
        "finished_naturally": finished_naturally,
    }


print("\nRuntime rebuilt successfully")
print("Model:", MODEL_ID)
print("Model class:", type(model).__name__)
print("Pilot questions:", len(pilot_df))
print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB",
)
print("Ready for baseline experiment")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.3 MB/s eta 0:00:00
GPU: Tesla T4


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]


Runtime rebuilt successfully
Model: Qwen/Qwen3.5-4B
Model class: Qwen3_5ForConditionalGeneration
Pilot questions: 20
GPU memory allocated: 3.08 GB
Ready for baseline experiment


In [3]:
import time
import pandas as pd
from tqdm.auto import tqdm

baseline_results = []
experiment_start = time.perf_counter()

for _, row in tqdm(
    pilot_df.iterrows(),
    total=len(pilot_df),
    desc="Running strict baseline",
):
    result = solve_strict_non_thinking(
        question=row["question"],
        max_new_tokens=128,
    )

    correct = answers_match(
        result["predicted_answer"],
        row["reference_answer"],
    )

    baseline_results.append(
        {
            "pilot_position": int(row["pilot_position"]),
            "gsm8k_index": int(row["gsm8k_index"]),
            "question": row["question"],
            "reference_answer": row["reference_answer"],
            "predicted_answer": result["predicted_answer"],
            "correct": correct,
            "generated_tokens": result["generated_tokens"],
            "elapsed_seconds": result["elapsed_seconds"],
            "finished_naturally": result["finished_naturally"],
            "raw_response": result["raw_response"],
        }
    )

    pd.DataFrame(baseline_results).to_csv(
        "/content/strict_baseline_results.csv",
        index=False,
    )

baseline_results_df = pd.DataFrame(baseline_results)

total_runtime = time.perf_counter() - experiment_start
accuracy = baseline_results_df["correct"].mean()
average_tokens = baseline_results_df["generated_tokens"].mean()
average_time = baseline_results_df["elapsed_seconds"].mean()

print("\nStrict non-thinking baseline completed")
print("Questions evaluated:", len(baseline_results_df))
print("Correct answers:", int(baseline_results_df["correct"].sum()))
print("Accuracy:", f"{accuracy:.1%}")
print("Average generated tokens:", f"{average_tokens:.1f}")
print("Average time per question:", f"{average_time:.2f} seconds")
print("Total experiment time:", f"{total_runtime / 60:.2f} minutes")
print(
    "Naturally completed:",
    f"{baseline_results_df['finished_naturally'].mean():.1%}",
)

display(
    baseline_results_df[
        [
            "pilot_position",
            "gsm8k_index",
            "reference_answer",
            "predicted_answer",
            "correct",
            "generated_tokens",
            "elapsed_seconds",
        ]
    ]
)

Running strict baseline:   0%|          | 0/20 [00:00<?, ?it/s]


Strict non-thinking baseline completed
Questions evaluated: 20
Correct answers: 3
Accuracy: 15.0%
Average generated tokens: 9.5
Average time per question: 1.45 seconds
Total experiment time: 0.49 minutes
Naturally completed: 100.0%


,pilot_position,gsm8k_index,reference_answer,predicted_answer,correct,generated_tokens,elapsed_seconds
0,0,5,64,68,False,9,4.176789
1,1,165,77,153,False,10,1.265178
2,2,210,10,10,True,9,1.548468
3,3,226,33,20,False,9,1.597773
4,4,243,250,150,False,10,1.240187
5,5,457,45,55,False,9,1.248243
6,6,491,16,16,True,9,1.219632
7,7,654,2,4,False,8,1.076457
8,8,861,30,1,False,8,1.117595
9,9,903,16,17.5,False,11,1.107785


In [4]:
import gc
import re
import time
import torch
import pandas as pd
from tqdm.auto import tqdm


THINKING_SYSTEM_PROMPT = (
    "Solve the mathematics problem carefully. "
    "Use concise reasoning, check the interpretation and arithmetic, "
    "and determine the final numeric answer."
)

FINALIZER_SYSTEM_PROMPT = (
    "You are a strict mathematical answer extractor. "
    "Use the supplied problem and reasoning trace to determine the "
    "final numeric answer. Return exactly one line in the format "
    "\\boxed{number}. Do not include explanation or additional text."
)


def run_thinking_with_finalizer(
    question,
    reasoning_budget=512,
    seed=2026,
):
    # ---------------------------------------------------------------
    # Stage 1: Thinking
    # ---------------------------------------------------------------

    thinking_messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": THINKING_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question,
                }
            ],
        },
    ]

    thinking_inputs = processor.apply_chat_template(
        thinking_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=True,
    ).to(model.device)

    thinking_input_length = thinking_inputs["input_ids"].shape[-1]

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.cuda.synchronize()
    thinking_start = time.perf_counter()

    with torch.inference_mode():
        thinking_outputs = model.generate(
            **thinking_inputs,
            max_new_tokens=reasoning_budget,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    thinking_seconds = time.perf_counter() - thinking_start

    thinking_tokens = thinking_outputs[0][thinking_input_length:]

    reasoning_trace = processor.decode(
        thinking_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    eos_ids = model.generation_config.eos_token_id

    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    elif eos_ids is None:
        eos_ids = []

    reasoning_finished_naturally = (
        len(thinking_tokens) > 0
        and int(thinking_tokens[-1]) in eos_ids
    )

    # ---------------------------------------------------------------
    # Stage 2: Deterministic finalizer
    # ---------------------------------------------------------------

    finalizer_messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": FINALIZER_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Problem:\n{question}\n\n"
                        f"Reasoning trace:\n{reasoning_trace}\n\n"
                        "Extract the final numeric answer."
                    ),
                }
            ],
        },
    ]

    finalizer_inputs = processor.apply_chat_template(
        finalizer_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    finalizer_input_length = finalizer_inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    finalizer_start = time.perf_counter()

    with torch.inference_mode():
        finalizer_outputs = model.generate(
            **finalizer_inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    finalizer_seconds = time.perf_counter() - finalizer_start

    finalizer_tokens = finalizer_outputs[0][finalizer_input_length:]

    finalizer_response = processor.decode(
        finalizer_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    boxed_answers = re.findall(
        r"\\boxed\{([^{}]+)\}",
        finalizer_response,
    )

    predicted_answer = (
        boxed_answers[-1]
        if boxed_answers
        else None
    )

    result = {
        "predicted_answer": predicted_answer,
        "reasoning_trace": reasoning_trace,
        "finalizer_response": finalizer_response,
        "reasoning_tokens": len(thinking_tokens),
        "finalizer_tokens": len(finalizer_tokens),
        "combined_tokens": (
            len(thinking_tokens) + len(finalizer_tokens)
        ),
        "reasoning_seconds": thinking_seconds,
        "finalizer_seconds": finalizer_seconds,
        "combined_seconds": (
            thinking_seconds + finalizer_seconds
        ),
        "reasoning_finished_naturally": (
            reasoning_finished_naturally
        ),
    }

    del (
        thinking_inputs,
        thinking_outputs,
        finalizer_inputs,
        finalizer_outputs,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return result


fixed_thinking_results = []
experiment_start = time.perf_counter()

for _, row in tqdm(
    pilot_df.iterrows(),
    total=len(pilot_df),
    desc="Running fixed 512-token thinking",
):
    pilot_position = int(row["pilot_position"])

    result = run_thinking_with_finalizer(
        question=row["question"],
        reasoning_budget=512,
        seed=2026 + pilot_position,
    )

    correct = answers_match(
        result["predicted_answer"],
        row["reference_answer"],
    )

    fixed_thinking_results.append(
        {
            "pilot_position": pilot_position,
            "gsm8k_index": int(row["gsm8k_index"]),
            "question": row["question"],
            "reference_answer": row["reference_answer"],
            "predicted_answer": result["predicted_answer"],
            "correct": correct,
            "reasoning_tokens": result["reasoning_tokens"],
            "finalizer_tokens": result["finalizer_tokens"],
            "combined_tokens": result["combined_tokens"],
            "reasoning_seconds": result["reasoning_seconds"],
            "finalizer_seconds": result["finalizer_seconds"],
            "combined_seconds": result["combined_seconds"],
            "reasoning_finished_naturally": (
                result["reasoning_finished_naturally"]
            ),
            "reasoning_trace": result["reasoning_trace"],
            "finalizer_response": result["finalizer_response"],
        }
    )

    pd.DataFrame(fixed_thinking_results).to_csv(
        "/content/fixed_512_thinking_results.csv",
        index=False,
    )


fixed_thinking_df = pd.DataFrame(fixed_thinking_results)

total_runtime = time.perf_counter() - experiment_start

print("\nFixed 512-token thinking experiment completed")
print("Questions evaluated:", len(fixed_thinking_df))
print(
    "Correct answers:",
    int(fixed_thinking_df["correct"].sum()),
)
print(
    "Accuracy:",
    f"{fixed_thinking_df['correct'].mean():.1%}",
)
print(
    "Average combined tokens:",
    f"{fixed_thinking_df['combined_tokens'].mean():.1f}",
)
print(
    "Average time per question:",
    f"{fixed_thinking_df['combined_seconds'].mean():.2f} seconds",
)
print(
    "Total experiment time:",
    f"{total_runtime / 60:.2f} minutes",
)
print(
    "Reasoning completed naturally:",
    f"{fixed_thinking_df['reasoning_finished_naturally'].mean():.1%}",
)

display(
    fixed_thinking_df[
        [
            "pilot_position",
            "gsm8k_index",
            "reference_answer",
            "predicted_answer",
            "correct",
            "reasoning_tokens",
            "finalizer_tokens",
            "combined_seconds",
        ]
    ]
)

Running fixed 512-token thinking:   0%|          | 0/20 [00:00<?, ?it/s]


Fixed 512-token thinking experiment completed
Questions evaluated: 20
Correct answers: 16
Accuracy: 80.0%
Average combined tokens: 521.6
Average time per question: 47.00 seconds
Total experiment time: 15.78 minutes
Reasoning completed naturally: 0.0%


,pilot_position,gsm8k_index,reference_answer,predicted_answer,correct,reasoning_tokens,finalizer_tokens,combined_seconds
0,0,5,64,64,True,512,9,49.876670
1,1,165,77,77,True,512,9,47.117449
2,2,210,10,10,True,512,9,47.570270
3,3,226,33,33,True,512,9,46.722549
4,4,243,250,250,True,512,10,46.952842
5,5,457,45,45,True,512,9,46.879181
6,6,491,16,16,True,512,9,47.153699
7,7,654,2,2,True,512,8,45.970675
8,8,861,30,18,False,512,9,46.292950
9,9,903,16,16,True,512,9,46.506033


In [5]:
import gc
import re
import time
import torch
import pandas as pd
from tqdm.auto import tqdm


ADAPTIVE_THINKING_SYSTEM_PROMPT = (
    "Solve the mathematics problem carefully. "
    "Use concise reasoning, verify the interpretation and arithmetic, "
    "and determine the final numeric answer."
)

ADAPTIVE_FINALIZER_SYSTEM_PROMPT = (
    "You are a strict mathematical answer extractor. "
    "Use the supplied problem and reasoning trace to determine the "
    "final numeric answer. Return exactly one line in the format "
    "\\boxed{number}. Do not include explanation or additional text."
)


def run_reasoning_candidate(
    question,
    reasoning_budget,
    seed,
):
    thinking_messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": ADAPTIVE_THINKING_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question,
                }
            ],
        },
    ]

    thinking_inputs = processor.apply_chat_template(
        thinking_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=True,
    ).to(model.device)

    thinking_input_length = thinking_inputs["input_ids"].shape[-1]

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.cuda.synchronize()
    thinking_start = time.perf_counter()

    with torch.inference_mode():
        thinking_outputs = model.generate(
            **thinking_inputs,
            max_new_tokens=reasoning_budget,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    thinking_seconds = time.perf_counter() - thinking_start

    thinking_tokens = thinking_outputs[0][thinking_input_length:]

    reasoning_trace = processor.decode(
        thinking_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    eos_ids = model.generation_config.eos_token_id

    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    elif eos_ids is None:
        eos_ids = []

    reasoning_finished_naturally = (
        len(thinking_tokens) > 0
        and int(thinking_tokens[-1]) in eos_ids
    )

    finalizer_messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": ADAPTIVE_FINALIZER_SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Problem:\n{question}\n\n"
                        f"Reasoning trace:\n{reasoning_trace}\n\n"
                        "Extract the final numeric answer."
                    ),
                }
            ],
        },
    ]

    finalizer_inputs = processor.apply_chat_template(
        finalizer_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    finalizer_input_length = finalizer_inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    finalizer_start = time.perf_counter()

    with torch.inference_mode():
        finalizer_outputs = model.generate(
            **finalizer_inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    finalizer_seconds = time.perf_counter() - finalizer_start

    finalizer_tokens = finalizer_outputs[0][finalizer_input_length:]

    finalizer_response = processor.decode(
        finalizer_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    boxed_answers = re.findall(
        r"\\boxed\{([^{}]+)\}",
        finalizer_response,
    )

    predicted_answer = boxed_answers[-1] if boxed_answers else None

    result = {
        "predicted_answer": predicted_answer,
        "reasoning_trace": reasoning_trace,
        "finalizer_response": finalizer_response,
        "reasoning_tokens": len(thinking_tokens),
        "finalizer_tokens": len(finalizer_tokens),
        "combined_tokens": len(thinking_tokens) + len(finalizer_tokens),
        "reasoning_seconds": thinking_seconds,
        "finalizer_seconds": finalizer_seconds,
        "combined_seconds": thinking_seconds + finalizer_seconds,
        "reasoning_finished_naturally": reasoning_finished_naturally,
    }

    del thinking_inputs, thinking_outputs, finalizer_inputs, finalizer_outputs
    gc.collect()
    torch.cuda.empty_cache()

    return result


adaptive_results = []
experiment_start = time.perf_counter()

for _, row in tqdm(
    pilot_df.iterrows(),
    total=len(pilot_df),
    desc="Running adaptive policy",
):
    pilot_position = int(row["pilot_position"])
    question = row["question"]
    reference_answer = row["reference_answer"]

    candidate_a = run_reasoning_candidate(
        question=question,
        reasoning_budget=128,
        seed=3000 + pilot_position * 10 + 1,
    )

    candidate_b = run_reasoning_candidate(
        question=question,
        reasoning_budget=128,
        seed=3000 + pilot_position * 10 + 2,
    )

    agreed = (
        candidate_a["predicted_answer"] is not None
        and candidate_b["predicted_answer"] is not None
        and answers_match(
            candidate_a["predicted_answer"],
            candidate_b["predicted_answer"],
        )
    )

    if agreed:
        final_prediction = candidate_a["predicted_answer"]
        escalated = False
        escalation_result = None
        total_tokens = (
            candidate_a["combined_tokens"]
            + candidate_b["combined_tokens"]
        )
        total_seconds = (
            candidate_a["combined_seconds"]
            + candidate_b["combined_seconds"]
        )
    else:
        escalation_result = run_reasoning_candidate(
            question=question,
            reasoning_budget=512,
            seed=3000 + pilot_position * 10 + 3,
        )
        final_prediction = escalation_result["predicted_answer"]
        escalated = True
        total_tokens = (
            candidate_a["combined_tokens"]
            + candidate_b["combined_tokens"]
            + escalation_result["combined_tokens"]
        )
        total_seconds = (
            candidate_a["combined_seconds"]
            + candidate_b["combined_seconds"]
            + escalation_result["combined_seconds"]
        )

    correct = answers_match(final_prediction, reference_answer)

    adaptive_results.append(
        {
            "pilot_position": pilot_position,
            "gsm8k_index": int(row["gsm8k_index"]),
            "question": question,
            "reference_answer": reference_answer,
            "candidate_a_answer": candidate_a["predicted_answer"],
            "candidate_b_answer": candidate_b["predicted_answer"],
            "agreed_early": agreed,
            "escalated": escalated,
            "final_predicted_answer": final_prediction,
            "correct": correct,
            "candidate_a_tokens": candidate_a["combined_tokens"],
            "candidate_b_tokens": candidate_b["combined_tokens"],
            "escalation_tokens": (
                escalation_result["combined_tokens"]
                if escalation_result is not None
                else 0
            ),
            "total_tokens": total_tokens,
            "candidate_a_seconds": candidate_a["combined_seconds"],
            "candidate_b_seconds": candidate_b["combined_seconds"],
            "escalation_seconds": (
                escalation_result["combined_seconds"]
                if escalation_result is not None
                else 0.0
            ),
            "total_seconds": total_seconds,
        }
    )

    pd.DataFrame(adaptive_results).to_csv(
        "/content/adaptive_policy_results.csv",
        index=False,
    )


adaptive_df = pd.DataFrame(adaptive_results)

total_runtime = time.perf_counter() - experiment_start

print("\nAdaptive policy experiment completed")
print("Questions evaluated:", len(adaptive_df))
print("Correct answers:", int(adaptive_df["correct"].sum()))
print("Accuracy:", f"{adaptive_df['correct'].mean():.1%}")
print("Early agreement rate:", f"{adaptive_df['agreed_early'].mean():.1%}")
print("Escalation rate:", f"{adaptive_df['escalated'].mean():.1%}")
print("Average total tokens:", f"{adaptive_df['total_tokens'].mean():.1f}")
print("Average time per question:", f"{adaptive_df['total_seconds'].mean():.2f} seconds")
print("Total experiment time:", f"{total_runtime / 60:.2f} minutes")

display(
    adaptive_df[
        [
            "pilot_position",
            "gsm8k_index",
            "reference_answer",
            "candidate_a_answer",
            "candidate_b_answer",
            "agreed_early",
            "escalated",
            "final_predicted_answer",
            "correct",
            "total_tokens",
            "total_seconds",
        ]
    ]
)

Running adaptive policy:   0%|          | 0/20 [00:00<?, ?it/s]


Adaptive policy experiment completed
Questions evaluated: 20
Correct answers: 12
Accuracy: 60.0%
Early agreement rate: 40.0%
Escalation rate: 60.0%
Average total tokens: 588.4
Average time per question: 54.27 seconds
Total experiment time: 18.37 minutes


,pilot_position,gsm8k_index,reference_answer,candidate_a_answer,candidate_b_answer,agreed_early,escalated,final_predicted_answer,correct,total_tokens,total_seconds
0,0,5,64,60,68,False,True,38,False,795,73.835055
1,1,165,77,177,177,True,False,177,False,276,25.116941
2,2,210,10,10,10,True,False,10,True,274,25.011170
3,3,226,33,10,20,False,True,33,True,795,71.942801
4,4,243,250,200,500,False,True,250,True,798,71.822817
5,5,457,45,65,40,False,True,45,True,795,71.664797
6,6,491,16,16,16,True,False,16,True,274,25.005174
7,7,654,2,1,4,False,True,2,True,792,70.697841
8,8,861,30,12,24,False,True,30,True,795,70.537730
9,9,903,16,19.5,14.5,False,True,16,True,799,70.204250


In [6]:
import os
import zipfile
import pandas as pd
from google.colab import files

output_directory = "/content/adaptive_reasoning_results"
os.makedirs(output_directory, exist_ok=True)

# Save complete raw results.
baseline_results_df.to_csv(
    f"{output_directory}/strict_non_thinking_results.csv",
    index=False,
)

fixed_thinking_df.to_csv(
    f"{output_directory}/fixed_512_thinking_results.csv",
    index=False,
)

adaptive_df.to_csv(
    f"{output_directory}/adaptive_policy_results.csv",
    index=False,
)

# Create a comparison summary.
comparison_summary = pd.DataFrame(
    [
        {
            "strategy": "Strict non-thinking",
            "questions": len(baseline_results_df),
            "correct_answers": int(
                baseline_results_df["correct"].sum()
            ),
            "accuracy": baseline_results_df["correct"].mean(),
            "average_generated_tokens": (
                baseline_results_df["generated_tokens"].mean()
            ),
            "average_seconds": (
                baseline_results_df["elapsed_seconds"].mean()
            ),
        },
        {
            "strategy": "Fixed 512-token thinking",
            "questions": len(fixed_thinking_df),
            "correct_answers": int(
                fixed_thinking_df["correct"].sum()
            ),
            "accuracy": fixed_thinking_df["correct"].mean(),
            "average_generated_tokens": (
                fixed_thinking_df["combined_tokens"].mean()
            ),
            "average_seconds": (
                fixed_thinking_df["combined_seconds"].mean()
            ),
        },
        {
            "strategy": "Naive adaptive agreement policy",
            "questions": len(adaptive_df),
            "correct_answers": int(
                adaptive_df["correct"].sum()
            ),
            "accuracy": adaptive_df["correct"].mean(),
            "average_generated_tokens": (
                adaptive_df["total_tokens"].mean()
            ),
            "average_seconds": (
                adaptive_df["total_seconds"].mean()
            ),
        },
    ]
)

comparison_summary.to_csv(
    f"{output_directory}/strategy_comparison_summary.csv",
    index=False,
)

# Save adaptive-policy diagnostics.
early_agreement_df = adaptive_df[
    adaptive_df["agreed_early"]
].copy()

escalated_df = adaptive_df[
    adaptive_df["escalated"]
].copy()

adaptive_diagnostics = pd.DataFrame(
    [
        {
            "metric": "Early agreements",
            "value": len(early_agreement_df),
        },
        {
            "metric": "Correct early agreements",
            "value": int(early_agreement_df["correct"].sum()),
        },
        {
            "metric": "Incorrect early agreements",
            "value": int((~early_agreement_df["correct"]).sum()),
        },
        {
            "metric": "Escalated questions",
            "value": len(escalated_df),
        },
        {
            "metric": "Correct after escalation",
            "value": int(escalated_df["correct"].sum()),
        },
        {
            "metric": "Incorrect after escalation",
            "value": int((~escalated_df["correct"]).sum()),
        },
    ]
)

adaptive_diagnostics.to_csv(
    f"{output_directory}/adaptive_policy_diagnostics.csv",
    index=False,
)

# Create a plain-text research note.
research_note = f"""
Pilot experiment summary
========================

Model:
Qwen/Qwen3.5-4B, 4-bit quantized

Benchmark:
20 reproducibly sampled GSM8K test questions

Strict non-thinking baseline:
Accuracy: {baseline_results_df['correct'].mean():.1%}
Average tokens: {baseline_results_df['generated_tokens'].mean():.1f}
Average time: {baseline_results_df['elapsed_seconds'].mean():.2f} seconds

Fixed 512-token thinking:
Accuracy: {fixed_thinking_df['correct'].mean():.1%}
Average tokens: {fixed_thinking_df['combined_tokens'].mean():.1f}
Average time: {fixed_thinking_df['combined_seconds'].mean():.2f} seconds

Naive adaptive agreement policy:
Accuracy: {adaptive_df['correct'].mean():.1%}
Average tokens: {adaptive_df['total_tokens'].mean():.1f}
Average time: {adaptive_df['total_seconds'].mean():.2f} seconds
Early agreement rate: {adaptive_df['agreed_early'].mean():.1%}

Initial finding:
Agreement between two short reasoning samples was not a reliable
confidence measure. Five of eight early agreements were incorrect,
showing that sampled reasoning paths can share correlated errors.
The naive adaptive policy therefore consumed more computation while
achieving lower accuracy than the fixed 512-token strategy.
""".strip()

with open(
    f"{output_directory}/pilot_research_note.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(research_note)

# Package everything into one ZIP file.
zip_path = "/content/adaptive_test_time_reasoning_results.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    for filename in os.listdir(output_directory):
        full_path = os.path.join(output_directory, filename)
        zip_file.write(
            full_path,
            arcname=filename,
        )

print("Backup created successfully")
print("ZIP file:", zip_path)
print("\nFiles included:")

for filename in sorted(os.listdir(output_directory)):
    print("-", filename)

files.download(zip_path)

Backup created successfully
ZIP file: /content/adaptive_test_time_reasoning_results.zip

Files included:
- adaptive_policy_diagnostics.csv
- adaptive_policy_results.csv
- fixed_512_thinking_results.csv
- pilot_research_note.txt
- strategy_comparison_summary.csv
- strict_non_thinking_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>